# 🔬 BioHub Cell Tracking — Kaggle Standalone Submission Notebook

**Competition:** [Biohub - Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development)

**Score Formula:** `edge_jaccard + 0.1 × division_jaccard`

### Pipeline Overview (100% Self-Contained — No External Dataset Required)
1. **Segmentation** — Cellpose 3D (cyto3 model) with Gaussian blob-detector fallback
2. **Post-processing** — Size filtering (50–50k voxels) + sequential relabelling
3. **Tracking** — Hungarian bipartite matching (10 µm cutoff, volume-cost enabled) + Gap-2 bridging (0.9× confidence penalty)
4. **Division Detection** — Orphan-pair volume conservation and spatial proximity classifier
5. **Submission** — 10-column CSV exporter (nodes + edges matching Kaggle competition schema)

In [2]:
# ── Cell 1: Imports and Core Data Structures ───────────────────────────────────
import csv
import logging
import pathlib
import sys
from dataclasses import dataclass, field
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Set, Tuple

import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)


@dataclass
class Cell:
    """Detected cell instance at a single frame."""
    id: int
    frame: int
    centroid: np.ndarray
    centroid_um: Optional[np.ndarray]
    volume: float
    features: Dict[str, float] = field(default_factory=dict)

    def distance_to(self, other: "Cell", use_um: bool = True) -> float:
        a = self.centroid_um if use_um and self.centroid_um is not None else self.centroid
        b = other.centroid_um if use_um and other.centroid_um is not None else other.centroid
        return float(np.linalg.norm(a - b))


@dataclass
class DivisionEvent:
    """Detected cell division event."""
    parent_id: int
    parent_frame: int
    child1_id: int
    child2_id: int
    division_frame: int
    confidence: float


def open_volume(path: str | pathlib.Path):
    """Open Zarr v3 volume and return 4D (T, Z, Y, X) array."""
    import zarr
    root = zarr.open(str(path), mode="r")
    array = root["0"] if "0" in root else root
    if len(array.shape) != 4:
        raise ValueError(f"Expected a 4D volume, got shape {array.shape}")
    return array


def iter_frames(path: str | pathlib.Path) -> Iterator[Tuple[int, np.ndarray]]:
    """Iterate through frames one by one to keep memory consumption low."""
    volume = open_volume(path)
    for frame_index in range(volume.shape[0]):
        yield frame_index, np.asarray(volume[frame_index])


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# ── Cell 2: 3D Segmentation and Post-processing ───────────────────────────────
from skimage.feature import blob_log
from skimage.filters import gaussian
from skimage.measure import regionprops
from skimage.segmentation import relabel_sequential, watershed


def postprocess_labels(labels: np.ndarray, min_volume: int = 50, max_volume: int = 50_000, remove_border: bool = False) -> np.ndarray:
    out = labels.copy()
    for region in regionprops(labels):
        if region.area < min_volume or region.area > max_volume:
            out[labels == region.label] = 0
    if remove_border:
        border_ids = set(np.unique(labels[0])) | set(np.unique(labels[-1]))
        border_ids |= set(np.unique(labels[:, :1, :])) | set(np.unique(labels[:, -1:, :]))
        border_ids.discard(0)
        for bid in border_ids:
            out[labels == bid] = 0
    out, _, _ = relabel_sequential(out)
    return out


class CellSegmenter:
    """3D Instance Segmenter with Cellpose cyto3 and Gaussian blob fallback."""
    def __init__(
        self,
        method: str = "cellpose",
        diameter: float = 12.0,
        do_3D: bool = True,
        anisotropy: float = 4.0,
        flow_threshold: float = 0.4,
        cellprob_threshold: float = 0.0,
        min_size: int = 50,
        max_volume: int = 50_000,
        channels: Tuple[int, int] = (0, 0),
        voxel_size_um: Tuple[float, float, float] = (1.625, 0.40625, 0.40625),
        model_type: str = "cyto3",
        remove_border: bool = False,
    ):
        self.method = method
        self.diameter = diameter
        self.do_3D = do_3D
        self.anisotropy = anisotropy
        self.flow_threshold = flow_threshold
        self.cellprob_threshold = cellprob_threshold
        self.min_size = min_size
        self.max_volume = max_volume
        self.channels = list(channels)
        self.voxel_size_um = voxel_size_um
        self.model_type = model_type
        self.remove_border = remove_border

    def segment_frame(self, img: np.ndarray, frame_idx: int = 0) -> Tuple[np.ndarray, List[Cell]]:
        if self.method == "cellpose":
            try:
                from cellpose import models
                model = models.CellposeModel(gpu=True, model_type=self.model_type)
                masks, _, _ = model.eval(
                    img,
                    diameter=self.diameter,
                    channels=self.channels,
                    do_3D=self.do_3D,
                    anisotropy=self.anisotropy,
                    flow_threshold=self.flow_threshold,
                    cellprob_threshold=self.cellprob_threshold,
                    min_size=self.min_size,
                )
                raw = masks.astype(np.int32)
            except Exception as e:
                logger.warning(f"Cellpose fallback to blob: {e}")
                raw = self._blob_segment(img)
        else:
            raw = self._blob_segment(img)

        labels = postprocess_labels(raw, min_volume=self.min_size, max_volume=self.max_volume, remove_border=self.remove_border)
        cells = []
        for region in regionprops(labels):
            centroid_vox = np.array(region.centroid, dtype=float)
            centroid_um = centroid_vox * np.array(self.voxel_size_um)
            cells.append(Cell(id=int(region.label), frame=frame_idx, centroid=centroid_vox, centroid_um=centroid_um, volume=float(region.area)))
        return labels, cells

    def _blob_segment(self, img: np.ndarray) -> np.ndarray:
        img_norm = img.astype(float)
        img_norm -= img_norm.min()
        if img_norm.max() > 0:
            img_norm /= img_norm.max()
        sigma_xy = self.diameter / 4.0
        sigma_z = sigma_xy / self.anisotropy
        smoothed = gaussian(img_norm, sigma=(sigma_z, sigma_xy, sigma_xy))
        blobs = blob_log(smoothed, min_sigma=max(1.0, self.diameter / 8), max_sigma=self.diameter / 2, num_sigma=5, threshold=0.05)
        if len(blobs) == 0:
            return np.zeros(img.shape, dtype=np.int32)
        seeds = np.zeros(img.shape, dtype=np.int32)
        for i, (z, y, x, _) in enumerate(blobs):
            zz, yy, xx = int(round(z)), int(round(y)), int(round(x))
            if 0 <= zz < img.shape[0] and 0 <= yy < img.shape[1] and 0 <= xx < img.shape[2]:
                seeds[zz, yy, xx] = i + 1
        return watershed(-smoothed, seeds, mask=smoothed > 0.05).astype(np.int32)


In [ ]:
# ── Cell 3: Hungarian Linker and Division Detector ────────────────────────────
from typing import Dict, Iterable, List, Optional, Set, Tuple
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


class HungarianLinker:
    """Frame-to-frame Hungarian algorithm linker with distance and volume cost."""
    def __init__(self, max_distance: float = 10.0, use_volume_cost: bool = True, volume_weight: float = 0.3):
        self.max_distance = max_distance
        self.use_volume_cost = use_volume_cost
        self.volume_weight = volume_weight

    def link(self, cells_t: List[Cell], cells_t1: List[Cell]) -> List[Tuple[int, int, float]]:
        if not cells_t or not cells_t1:
            return []
        c0 = np.array([c.centroid_um if c.centroid_um is not None else c.centroid for c in cells_t])
        c1 = np.array([c.centroid_um if c.centroid_um is not None else c.centroid for c in cells_t1])
        cost = cdist(c0, c1, metric="euclidean")

        if self.use_volume_cost:
            vol_t = np.array([c.volume for c in cells_t])[:, None]
            vol_t1 = np.array([c.volume for c in cells_t1])[None, :]
            vol_ratio = np.minimum(vol_t, vol_t1) / (np.maximum(vol_t, vol_t1) + 1e-8)
            vol_cost = (1.0 - vol_ratio) * self.max_distance
            cost = (1.0 - self.volume_weight) * cost + self.volume_weight * vol_cost

        cost_masked = cost.copy()
        cost_masked[cost >= self.max_distance] = self.max_distance * 10
        row_ind, col_ind = linear_sum_assignment(cost_masked)

        links = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < self.max_distance:
                conf = 1.0 - cost[r, c] / self.max_distance
                links.append((cells_t[r].id, cells_t1[c].id, float(conf)))
        return links


class DivisionDetector:
    """Detect cell division events from unlinked orphan cells."""
    def __init__(self, min_size_ratio: float = 0.3, max_size_ratio: float = 0.8, max_distance_um: float = 10.0, vol_balance_weight: float = 0.5, dist_score_weight: float = 0.5):
        self.min_size_ratio = min_size_ratio
        self.max_size_ratio = max_size_ratio
        self.max_distance_um = max_distance_um
        self.vol_balance_weight = vol_balance_weight
        self.dist_score_weight = dist_score_weight

    def detect(self, all_cells: Dict[int, List[Cell]], links: List[Tuple[int, int, int, float]]) -> List[DivisionEvent]:
        divisions = []
        frames = sorted(all_cells.keys())
        for i in range(len(frames) - 1):
            t, t1 = frames[i], frames[i+1]
            cells_t, cells_t1 = all_cells[t], all_cells[t1]
            linked_t = {l[1] for l in links if l[0] == t}
            linked_t1 = {l[2] for l in links if l[0] == t}
            orphans_t = [c for c in cells_t if c.id not in linked_t]
            orphans_t1 = [c for c in cells_t1 if c.id not in linked_t1]
            if not orphans_t or len(orphans_t1) < 2:
                continue

            for mother in orphans_t:
                candidates = [d for d in orphans_t1 if mother.distance_to(d, use_um=True) <= self.max_distance_um]
                if len(candidates) >= 2:
                    for i1 in range(len(candidates)):
                        for i2 in range(i1 + 1, len(candidates)):
                            d1, d2 = candidates[i1], candidates[i2]
                            conf = self._score(mother, d1, d2)
                            if conf > 0.5:
                                divisions.append(DivisionEvent(mother.id, t, d1.id, d2.id, t1, conf))
                                break
                        else:
                            continue
                        break
        return divisions

    def _score(self, mother: Cell, d1: Cell, d2: Cell) -> float:
        vol_err = abs(mother.volume - (d1.volume + d2.volume)) / (mother.volume + 1e-8)
        vol_score = 1.0 - min(1.0, vol_err)
        r1, r2 = d1.volume / (mother.volume + 1e-8), d2.volume / (mother.volume + 1e-8)
        sym_score = 1.0 if (self.min_size_ratio <= r1 <= self.max_size_ratio and self.min_size_ratio <= r2 <= self.max_size_ratio) else 0.0
        avg_dist = (mother.distance_to(d1) + mother.distance_to(d2)) / 2.0
        dist_score = 1.0 - (avg_dist / self.max_distance_um)
        return float((self.vol_balance_weight * vol_score * sym_score) + (self.dist_score_weight * dist_score))


In [ ]:
# ── Cell 4: Kaggle Submission CSV Exporter ─────────────────────────────────────
HEADER = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]

class SubmissionBuilder:
    """Serialize node detections and temporal edges into Kaggle submission CSV format."""
    def build_rows(self, dataset: str, all_cells: Dict[int, Sequence[Cell]], links: Iterable[Tuple[int, int, int, float]], divisions: Iterable[DivisionEvent] = ()) -> List[dict]:
        node_ids = {}
        rows = []
        next_node_id = 1
        for frame in sorted(all_cells):
            for cell in sorted(all_cells[frame], key=lambda c: c.id):
                node_ids[(frame, cell.id)] = next_node_id
                z, y, x = (int(round(v)) for v in cell.centroid)
                rows.append({"dataset": dataset, "row_type": "node", "node_id": next_node_id, "t": frame, "z": z, "y": y, "x": x, "source_id": -1, "target_id": -1})
                next_node_id += 1

        edge_keys = set()
        edges = []
        for frame, src, tgt, _ in links:
            s_key, t_key = (frame, src), (frame + 1, tgt)
            k = (s_key, t_key)
            if s_key in node_ids and t_key in node_ids and k not in edge_keys:
                edge_keys.add(k)
                edges.append(k)

        for div in divisions:
            p_key = (div.parent_frame, div.parent_id)
            for ch in (div.child1_id, div.child2_id):
                c_key = (div.division_frame, ch)
                k = (p_key, c_key)
                if p_key in node_ids and c_key in node_ids and k not in edge_keys:
                    edge_keys.add(k)
                    edges.append(k)

        for (s_frame, s_cell), (t_frame, t_cell) in edges:
            rows.append({"dataset": dataset, "row_type": "edge", "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1, "source_id": node_ids[(s_frame, s_cell)], "target_id": node_ids[(t_frame, t_cell)]})
        return rows

    def write(self, rows: Iterable[dict], output_path: str | pathlib.Path) -> None:
        output_path = pathlib.Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", newline="", encoding="utf-8") as h:
            writer = csv.DictWriter(h, fieldnames=HEADER)
            writer.writeheader()
            for r_id, row in enumerate(rows):
                writer.writerow({"id": r_id, **row})


In [ ]:
# ── Cell 5: Full Pipeline Execution and Submission Generation ─────────────────
ON_KAGGLE = pathlib.Path('/kaggle').exists()

if ON_KAGGLE:
    TEST_DIR = pathlib.Path('/kaggle/input/biohub-cell-tracking-during-development/test')
    OUTPUT   = pathlib.Path('/kaggle/working/submission.csv')
else:
    TEST_DIR = pathlib.Path('../data/test')
    OUTPUT   = pathlib.Path('submission.csv')

def find_test_samples(test_dir: pathlib.Path) -> List[pathlib.Path]:
    """Robustly locate all .zarr test samples under test_dir or /kaggle/input/."""
    if not test_dir.exists() and ON_KAGGLE:
        input_base = pathlib.Path('/kaggle/input')
        if input_base.exists():
            for p in input_base.rglob('test'):
                if p.is_dir():
                    test_dir = p
                    break

    if not test_dir.exists():
        return []

    samples = [p for p in test_dir.iterdir() if p.name.endswith('.zarr') or (p.is_dir() and not p.name.startswith('.'))]
    return sorted(samples)

def process_sample(sample_path: pathlib.Path, segmenter: CellSegmenter):
    all_cells = {}
    for frame_index, image in iter_frames(sample_path):
        _, cells = segmenter.segment_frame(image, frame_index)
        all_cells[frame_index] = cells

    linker = HungarianLinker(max_distance=10.0, use_volume_cost=True, volume_weight=0.3)
    links = []
    linked_sources = {f: set() for f in all_cells}
    linked_targets = {f: set() for f in all_cells}
    sorted_frames = sorted(all_cells)

    # Primary consecutive linking
    for frame in sorted_frames[:-1]:
        fl = linker.link(all_cells[frame], all_cells[frame + 1])
        for src, tgt, conf in fl:
            links.append((frame, src, tgt, conf))
            linked_sources[frame].add(src)
            linked_targets[frame + 1].add(tgt)

    # Gap-2 bridging (reconnect cells missing for 1 frame)
    for frame in sorted_frames[:-2]:
        t2 = frame + 2
        if t2 not in all_cells:
            continue
        orphan_src = [c for c in all_cells[frame] if c.id not in linked_sources[frame]]
        orphan_tgt = [c for c in all_cells[t2] if c.id not in linked_targets[t2]]
        if not orphan_src or not orphan_tgt:
            continue
        for src, tgt, conf in linker.link(orphan_src, orphan_tgt):
            links.append((frame, src, tgt, conf * 0.9))
            linked_sources[frame].add(src)
            linked_targets[t2].add(tgt)

    divisions = DivisionDetector(max_distance_um=10.0).detect(all_cells, links)
    return all_cells, links, divisions

# Run Pipeline
segmenter = CellSegmenter(method='cellpose', diameter=12.0, do_3D=True, anisotropy=4.0, model_type='cyto3')
builder = SubmissionBuilder()
all_rows = []

samples = find_test_samples(TEST_DIR)
print(f"Environment    : {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Test directory : {TEST_DIR}")
print(f"Found {len(samples)} test samples.")
for s in samples[:5]:
    print(f"  - {s.name}")

for sp in samples:
    logger.info(f"Processing {sp.name}...")
    cells, links, divisions = process_sample(sp, segmenter)
    rows = builder.build_rows(sp.stem, cells, links, divisions)
    all_rows.extend(rows)

if all_rows:
    builder.write(all_rows, OUTPUT)
    print(f"\n✅ SUCCESS: Wrote {len(all_rows):,} rows to {OUTPUT}")
else:
    print("\n⚠️ No test samples found. Verify competition data path.")


In [ ]:
# ── Cell 6: Preview and Schema Validation ──────────────────────────────────────
import pandas as pd

if OUTPUT.exists():
    df = pd.read_csv(OUTPUT)
    print("=" * 60)
    print(f"Total Rows   : {len(df):,}")
    print(f"Node Rows    : {(df.row_type == 'node').sum():,}")
    print(f"Edge Rows    : {(df.row_type == 'edge').sum():,}")
    print(f"Datasets     : {df.dataset.nunique()}")
    print(f"Columns      : {list(df.columns)}")
    print("=" * 60)
    
    # Schema assertions
    expected_cols = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    assert list(df.columns) == expected_cols, f"Schema mismatch! Expected {expected_cols}"
    assert df.row_type.isin(['node', 'edge']).all(), "Invalid row_type found!"
    print("\n✅ All schema assertions passed!")
    print("\nFirst 10 rows:")
    display(df.head(10))
else:
    print("No submission CSV found to validate.")
